# Business Performance Analytics

This project compares six technology companies on operating performance. The SQL calculates the metrics. The management analysis is at the end: an executive summary, one section for each company, a cross-company comparison, and a management attention table.

**Business question**

How is each company performing across growth, operations, profitability, cash generation, and financial health, and where should management attention be focused?

**Companies**

| Company | Sector |
|---|---|
| Microsoft | Software & Cloud |
| Apple | Consumer Technology |
| NVIDIA | Semiconductors |
| Adobe | Software |
| AMD | Semiconductors |
| CrowdStrike | Cybersecurity |

**Fiscal-year coverage**

Latest year is not the same calendar year for every company. Microsoft, NVIDIA, and CrowdStrike have a FY2026 10-K. AMD, Adobe, and Apple stop at FY2025. FY2025 is the last year available for all six companies. A FY2026 comparison drops AMD, Adobe, and Apple. Company analyses use each company's own latest year. Peer ranks stay on FY2025.

**How to read this notebook**

1. Growth and momentum
2. Profitability
3. Operating efficiency
4. Cash flow and cash conversion
5. Financial health
6. Peer benchmarking for FY2025
7. Executive summary and individual company analysis
8. Cross-company comparison and management attention summary

**Pipeline**

SEC 10-K filings, Python preparation, `financial_analytics.clean.company_financials`, SQL views, then the company analysis.

Views used: `growth_trend`, `performance`, `financial_health`, `peer_benchmarking`, and `executive_summary`.

## Data Preparation & Quality

### Business Question

Is the data complete enough to support the operating metrics?

Source table: `financial_analytics.clean.company_financials`

Missing values are left null. They are not filled with zero.

AMD Total Liabilities was missing in the source file. Where needed it is calculated as:

Total Liabilities = Total Assets − Stockholders’ Equity

`extract_sec_company_facts.py` writes `data/raw/sec_company_financials_raw.csv`, `data/processed/sec_company_financials_clean.csv`, and `data/processed/company_financials_wide.csv`. This notebook starts from `financial_analytics.clean.company_financials` and `financial_analytics.clean.sec_company_financials_clean`, which are the warehouse copies of those prepared files. The upload step is not in this notebook.


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS financial_analytics.data_quality;

CREATE OR REPLACE VIEW financial_analytics.data_quality.sec_company_financials_null_summary AS

SELECT
    COUNT(*) AS total_records,
    SUM(CASE WHEN company IS NULL THEN 1 ELSE 0 END) AS company_nulls,
    SUM(CASE WHEN ticker IS NULL THEN 1 ELSE 0 END) AS ticker_nulls,
    SUM(CASE WHEN statement IS NULL THEN 1 ELSE 0 END) AS statement_nulls,
    SUM(CASE WHEN fiscal_year IS NULL THEN 1 ELSE 0 END) AS fiscal_year_nulls,
    SUM(CASE WHEN Metric IS NULL THEN 1 ELSE 0 END) AS Metric_nulls,
    SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END) AS value_nulls,
    ROUND(SUM(CASE WHEN company IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS company_null_pct,
    ROUND(SUM(CASE WHEN ticker IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS ticker_null_pct,
    ROUND(SUM(CASE WHEN statement IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS statement_null_pct,
    ROUND(SUM(CASE WHEN fiscal_year IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS fiscal_year_null_pct,
    ROUND(SUM(CASE WHEN Metric IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Metric_null_pct,
    ROUND(SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS value_null_pct
FROM financial_analytics.clean.sec_company_financials_clean;


In [0]:
%sql
SELECT * FROM financial_analytics.data_quality.sec_company_financials_null_summary;


total_records,company_nulls,ticker_nulls,statement_nulls,fiscal_year_nulls,Metric_nulls,value_nulls,company_null_pct,ticker_null_pct,statement_null_pct,fiscal_year_null_pct,Metric_null_pct,value_null_pct
1657,0,0,0,0,0,0,0.00,0.00,0.00,0.00,0.00,0.00


In [0]:
%sql
Update financial_analytics.clean.company_financials

set Total_Liabilities = Total_Assets - Total_Stockholders_Equity
where Company = 'AMD'
and Total_Liabilities is null;


num_affected_rows
5


### Company-year quality checks

Each row is one control. Counts come from the clean table.


In [0]:
%sql
CREATE OR REPLACE VIEW financial_analytics.data_quality.company_year_quality AS
SELECT
    COUNT(*) AS total_company_years,
    SUM(CASE WHEN Revenue IS NULL THEN 1 ELSE 0 END) AS missing_revenue,
    SUM(CASE WHEN Operating_Income IS NULL THEN 1 ELSE 0 END) AS missing_operating_income,
    SUM(CASE WHEN Net_Income IS NULL THEN 1 ELSE 0 END) AS missing_net_income,
    SUM(CASE WHEN Operating_Expenses IS NULL THEN 1 ELSE 0 END) AS missing_operating_expenses,
    SUM(CASE WHEN Cash_Flow_from_Operating_Activities IS NULL THEN 1 ELSE 0 END) AS missing_operating_cash_flow,
    SUM(CASE WHEN Total_Current_Assets IS NULL THEN 1 ELSE 0 END) AS missing_current_assets,
    SUM(CASE WHEN Total_Current_Liabilities IS NULL THEN 1 ELSE 0 END) AS missing_current_liabilities,
    SUM(CASE WHEN Total_Assets IS NULL THEN 1 ELSE 0 END) AS missing_total_assets,
    SUM(CASE WHEN Total_Liabilities IS NULL THEN 1 ELSE 0 END) AS missing_total_liabilities,
    SUM(CASE WHEN Total_Stockholders_Equity IS NULL THEN 1 ELSE 0 END) AS missing_equity,
    SUM(CASE WHEN Total_Stockholders_Equity <= 0 THEN 1 ELSE 0 END) AS zero_or_negative_equity,
    SUM(CASE WHEN Short_Term_Debt IS NULL AND Long_Term_Debt IS NULL THEN 1 ELSE 0 END) AS missing_debt,
    SUM(CASE WHEN Total_Current_Liabilities = 0 THEN 1 ELSE 0 END) AS zero_current_liabilities
FROM financial_analytics.clean.company_financials;


In [0]:
%sql
SELECT *
FROM financial_analytics.data_quality.company_year_quality;


total_company_years,missing_revenue,missing_operating_income,missing_net_income,missing_operating_expenses,missing_operating_cash_flow,missing_current_assets,missing_current_liabilities,missing_total_assets,missing_total_liabilities,missing_equity,zero_or_negative_equity,missing_debt,zero_current_liabilities
33,0,0,0,4,0,0,0,0,0,0,0,0,0


## SQL views




In [0]:
%sql
CREATE OR REPLACE VIEW financial_analytics.analytics.growth_trend AS
WITH growth AS (
    SELECT
        Company,
        Fiscal_Year,
        Revenue,
        Net_Income,
        (Revenue - LAG(Revenue) OVER (PARTITION BY Company ORDER BY Fiscal_Year))
            / NULLIF(LAG(Revenue) OVER (PARTITION BY Company ORDER BY Fiscal_Year), 0) AS Growth_Rate
    FROM financial_analytics.clean.company_financials
),
growth_metrics AS (
    SELECT
        Company,
        Fiscal_Year,
        Revenue,
        Net_Income,
        Growth_Rate,
        LAG(Growth_Rate) OVER (PARTITION BY Company ORDER BY Fiscal_Year) AS Prior_Growth_Rate,
        Growth_Rate - LAG(Growth_Rate) OVER (PARTITION BY Company ORDER BY Fiscal_Year) AS Growth_Differential
    FROM growth
)
SELECT
    Company,
    Fiscal_Year,
    Revenue,
    Net_Income,
    round(Growth_Rate, 3) AS Growth_Rate,
    round(Prior_Growth_Rate, 3) AS Prior_Growth_Rate,
    round(Growth_Differential, 3) AS Growth_Differential,
    round(Net_Income / NULLIF(Revenue, 0), 3) AS Net_Income_Margin_pct
FROM growth_metrics


In [0]:
%sql
CREATE OR REPLACE VIEW financial_analytics.analytics.performance AS
WITH base AS (
    SELECT
        c.Company,
        c.Fiscal_Year,
        c.Revenue,
        c.Gross_Profit,
        c.Net_Income,
        c.Operating_Income,
        c.Operating_Expenses,
        c.Cash_Flow_from_Operating_Activities,
        c.Capital_Expenditures,
        LAG(c.Revenue) OVER (PARTITION BY c.Company ORDER BY c.Fiscal_Year) AS Prior_Revenue,
        LAG(c.Operating_Income) OVER (PARTITION BY c.Company ORDER BY c.Fiscal_Year) AS Prior_Operating_Income,
        LAG(c.Net_Income) OVER (PARTITION BY c.Company ORDER BY c.Fiscal_Year) AS Prior_Net_Income,
        LAG(c.Operating_Expenses) OVER (PARTITION BY c.Company ORDER BY c.Fiscal_Year) AS Prior_Opex,
        LAG(c.Cash_Flow_from_Operating_Activities) OVER (PARTITION BY c.Company ORDER BY c.Fiscal_Year) AS Prior_OCF
    FROM financial_analytics.clean.company_financials c
)
SELECT
    b.Company,
    b.Fiscal_Year,
    b.Revenue,
    b.Gross_Profit,
    b.Net_Income,
    b.Operating_Income,
    b.Operating_Expenses,
    b.Cash_Flow_from_Operating_Activities,
    b.Capital_Expenditures,
    gt.Growth_Rate,
    gt.Net_Income_Margin_pct,
    round(b.Gross_Profit / NULLIF(b.Revenue, 0), 3) AS Gross_Margin_pct,
    round(b.Operating_Income / NULLIF(b.Revenue, 0), 3) AS Operating_Margin_pct,
    round((b.Operating_Income - b.Prior_Operating_Income) / NULLIF(b.Prior_Operating_Income, 0), 3) AS Operating_Income_Growth,
    round((b.Net_Income - b.Prior_Net_Income) / NULLIF(b.Prior_Net_Income, 0), 3) AS Net_Income_Growth,
    round((b.Operating_Expenses - b.Prior_Opex) / NULLIF(b.Prior_Opex, 0), 3) AS Operating_Expense_Growth,
    round(b.Operating_Expenses / NULLIF(b.Revenue, 0), 3) AS Operating_Expense_to_Revenue,
    round(
        ((b.Revenue - b.Prior_Revenue) / NULLIF(b.Prior_Revenue, 0))
        - ((b.Operating_Expenses - b.Prior_Opex) / NULLIF(b.Prior_Opex, 0)),
        3
    ) AS Revenue_Growth_Minus_Opex_Growth,
    round(b.Cash_Flow_from_Operating_Activities / NULLIF(b.Revenue, 0), 3) AS Operating_Cash_Flow_Margin_pct,
    round(
        (b.Cash_Flow_from_Operating_Activities - b.Prior_OCF) / NULLIF(b.Prior_OCF, 0),
        3
    ) AS Operating_Cash_Flow_Growth,
    round(b.Cash_Flow_from_Operating_Activities / NULLIF(b.Net_Income, 0), 3) AS Cash_Conversion_Ratio,
    CASE
        WHEN b.Net_Income IS NULL OR b.Cash_Flow_from_Operating_Activities IS NULL THEN NULL
        WHEN b.Net_Income <= 0 THEN FALSE
        WHEN abs(b.Cash_Flow_from_Operating_Activities / b.Net_Income) > 5 THEN FALSE
        ELSE TRUE
    END AS Cash_Conversion_Meaningful,
    round(b.Cash_Flow_from_Operating_Activities - b.Capital_Expenditures, 3) AS Free_Cash_Flow,
    round(
        round(b.Cash_Flow_from_Operating_Activities - b.Capital_Expenditures, 3) / NULLIF(b.Revenue, 0),
        3
    ) AS Free_Cash_Flow_Margin_pct
FROM base b
LEFT JOIN financial_analytics.analytics.growth_trend gt
    ON b.Company = gt.Company
    AND b.Fiscal_Year = gt.Fiscal_Year


In [0]:
%sql
CREATE OR REPLACE VIEW financial_analytics.analytics.financial_health AS
WITH reported_liabilities AS (
    SELECT DISTINCT
        Company,
        Fiscal_Year
    FROM financial_analytics.clean.sec_company_financials_clean
    WHERE Metric = 'Total_Liabilities'
),
source AS (
    SELECT
        c.Company,
        c.Ticker,
        c.Fiscal_Year,
        c.Total_Assets,
        c.Total_Stockholders_Equity,
        c.Total_Current_Assets,
        c.Total_Current_Liabilities,
        c.Short_Term_Debt,
        c.Long_Term_Debt,
        c.Net_Income,
        c.Cash_Flow_from_Operating_Activities,
        CASE
            WHEN c.Company = 'AMD' AND r.Company IS NULL
                THEN COALESCE(c.Total_Liabilities, c.Total_Assets - c.Total_Stockholders_Equity)
            ELSE c.Total_Liabilities
        END AS Total_Liabilities,
        CASE
            WHEN c.Company = 'AMD' AND r.Company IS NULL THEN TRUE
            ELSE FALSE
        END AS Total_Liabilities_Imputed
    FROM financial_analytics.clean.company_financials c
    LEFT JOIN reported_liabilities r
        ON c.Company = r.Company
        AND c.Fiscal_Year = r.Fiscal_Year
),
with_prior AS (
    SELECT
        source.*,
        LAG(Total_Assets) OVER (PARTITION BY Ticker ORDER BY Fiscal_Year) AS Prior_Total_Assets,
        LAG(Total_Stockholders_Equity) OVER (PARTITION BY Ticker ORDER BY Fiscal_Year) AS Prior_Total_Equity
    FROM source
)
SELECT
    Company,
    Fiscal_Year,
    Total_Assets,
    Total_Liabilities,
    Total_Liabilities_Imputed,
    Total_Stockholders_Equity,
    Total_Current_Assets,
    Total_Current_Liabilities,
    round(Total_Current_Assets / NULLIF(Total_Current_Liabilities, 0), 3) AS Current_ratio,
    COALESCE(Short_Term_Debt, 0) + COALESCE(Long_Term_Debt, 0) AS Total_Debt,
    (Short_Term_Debt IS NOT NULL) AS Short_Term_Debt_Reported,
    (Long_Term_Debt IS NOT NULL) AS Long_Term_Debt_Reported,
    round(
        (COALESCE(Short_Term_Debt, 0) + COALESCE(Long_Term_Debt, 0))
            / NULLIF(Total_Stockholders_Equity, 0),
        3
    ) AS debt_to_equity_ratio,
    round(Total_Liabilities / NULLIF(Total_Assets, 0), 3) AS Liabilities_to_Assets,
    round(Net_Income / NULLIF(Total_Assets, 0), 3) AS ROA_Ending,
    round(Net_Income / NULLIF(Total_Stockholders_Equity, 0), 3) AS ROE_Ending,
    round(
        Net_Income / NULLIF((Total_Assets + Prior_Total_Assets) / 2, 0),
        3
    ) AS ROA_Avg,
    round(
        Net_Income / NULLIF((Total_Stockholders_Equity + Prior_Total_Equity) / 2, 0),
        3
    ) AS ROE_Avg,
    CASE WHEN Prior_Total_Assets IS NULL THEN TRUE ELSE FALSE END AS ROA_Uses_Ending_Balance,
    CASE WHEN Prior_Total_Equity IS NULL THEN TRUE ELSE FALSE END AS ROE_Uses_Ending_Balance,
    round(
        CASE
            WHEN Prior_Total_Assets IS NULL THEN Net_Income / NULLIF(Total_Assets, 0)
            ELSE Net_Income / NULLIF((Total_Assets + Prior_Total_Assets) / 2, 0)
        END,
        3
    ) AS ROA,
    round(
        CASE
            WHEN Prior_Total_Equity IS NULL THEN Net_Income / NULLIF(Total_Stockholders_Equity, 0)
            ELSE Net_Income / NULLIF((Total_Stockholders_Equity + Prior_Total_Equity) / 2, 0)
        END,
        3
    ) AS ROE,
    Cash_Flow_from_Operating_Activities
FROM with_prior


In [0]:
%sql
CREATE OR REPLACE VIEW financial_analytics.analytics.peer_benchmarking AS
WITH ranked_metrics AS (
    SELECT
        p.Company,
        p.Fiscal_Year,
        gt.Growth_Rate,
        p.Operating_Margin_pct,
        p.Net_Income_Margin_pct,
        p.Operating_Expense_to_Revenue,
        p.Operating_Cash_Flow_Margin_pct,
        fh.Current_ratio,
        fh.debt_to_equity_ratio,
        fh.ROA,
        fh.ROE,
        RANK() OVER (PARTITION BY p.Fiscal_Year ORDER BY gt.Growth_Rate DESC NULLS LAST) AS Growth_Rate_Rank,
        RANK() OVER (PARTITION BY p.Fiscal_Year ORDER BY p.Operating_Margin_pct DESC NULLS LAST) AS Operating_Margin_Rank,
        RANK() OVER (PARTITION BY p.Fiscal_Year ORDER BY p.Net_Income_Margin_pct DESC NULLS LAST) AS Net_Income_Margin_Rank,
        RANK() OVER (PARTITION BY p.Fiscal_Year ORDER BY p.Operating_Expense_to_Revenue ASC NULLS LAST) AS Operating_Expense_to_Revenue_Rank,
        RANK() OVER (PARTITION BY p.Fiscal_Year ORDER BY p.Operating_Cash_Flow_Margin_pct DESC NULLS LAST) AS Operating_Cash_Flow_Margin_Rank,
        RANK() OVER (PARTITION BY p.Fiscal_Year ORDER BY fh.Current_ratio DESC NULLS LAST) AS Current_Ratio_Rank,
        RANK() OVER (PARTITION BY p.Fiscal_Year ORDER BY fh.debt_to_equity_ratio ASC NULLS LAST) AS Debt_to_Equity_Rank,
        RANK() OVER (PARTITION BY p.Fiscal_Year ORDER BY fh.ROA DESC NULLS LAST) AS ROA_Rank,
        RANK() OVER (PARTITION BY p.Fiscal_Year ORDER BY fh.ROE DESC NULLS LAST) AS ROE_Rank
    FROM financial_analytics.analytics.performance p
    LEFT JOIN financial_analytics.analytics.financial_health fh
        ON p.Company = fh.Company
        AND p.Fiscal_Year = fh.Fiscal_Year
    LEFT JOIN financial_analytics.analytics.growth_trend gt
        ON p.Company = gt.Company
        AND p.Fiscal_Year = gt.Fiscal_Year
    WHERE p.Fiscal_Year = 2025
)
SELECT
    Company,
    Fiscal_Year,
    Growth_Rate,
    Growth_Rate_Rank,
    Operating_Margin_pct,
    Operating_Margin_Rank,
    Net_Income_Margin_pct,
    Net_Income_Margin_Rank,
    Operating_Expense_to_Revenue,
    Operating_Expense_to_Revenue_Rank,
    Operating_Cash_Flow_Margin_pct,
    Operating_Cash_Flow_Margin_Rank,
    Current_ratio,
    Current_Ratio_Rank,
    debt_to_equity_ratio,
    Debt_to_Equity_Rank,
    ROA,
    ROA_Rank,
    ROE,
    ROE_Rank,
    round((
        Growth_Rate_Rank
        + Operating_Margin_Rank
        + Net_Income_Margin_Rank
        + Operating_Expense_to_Revenue_Rank
        + Operating_Cash_Flow_Margin_Rank
        + Current_Ratio_Rank
        + Debt_to_Equity_Rank
        + ROA_Rank
        + ROE_Rank
    ) / 9.0, 3) AS Average_Rank
FROM ranked_metrics


In [0]:
%sql
CREATE OR REPLACE VIEW financial_analytics.analytics.executive_summary AS
WITH latest AS (
    SELECT Company, MAX(Fiscal_Year) AS Latest_Fiscal_Year
    FROM financial_analytics.analytics.performance
    GROUP BY Company
)
SELECT
    p.Company,
    p.Fiscal_Year,
    2025 AS Benchmark_Fiscal_Year,
    gt.Revenue,
    gt.Growth_Rate,
    gt.Prior_Growth_Rate,
    gt.Growth_Differential,
    p.Gross_Margin_pct,
    p.Operating_Margin_pct,
    p.Net_Income_Margin_pct,
    p.Operating_Income_Growth,
    p.Net_Income_Growth,
    p.Operating_Expense_Growth,
    p.Operating_Expense_to_Revenue,
    p.Revenue_Growth_Minus_Opex_Growth,
    p.Operating_Cash_Flow_Margin_pct,
    p.Operating_Cash_Flow_Growth,
    p.Cash_Conversion_Ratio,
    p.Cash_Conversion_Meaningful,
    p.Free_Cash_Flow,
    p.Free_Cash_Flow_Margin_pct,
    fh.Current_ratio,
    fh.debt_to_equity_ratio,
    fh.Liabilities_to_Assets,
    fh.ROA,
    fh.ROE,
    fh.ROA_Ending,
    fh.ROE_Ending,
    fh.ROA_Avg,
    fh.ROE_Avg,
    fh.ROA_Uses_Ending_Balance,
    fh.ROE_Uses_Ending_Balance,
    fh.Total_Liabilities_Imputed,
    fh.Short_Term_Debt_Reported,
    fh.Long_Term_Debt_Reported,
    pb.Growth_Rate_Rank AS FY2025_Growth_Rate_Rank,
    pb.Operating_Margin_Rank AS FY2025_Operating_Margin_Rank,
    pb.Net_Income_Margin_Rank AS FY2025_Net_Income_Margin_Rank,
    pb.Operating_Expense_to_Revenue_Rank AS FY2025_Operating_Expense_to_Revenue_Rank,
    pb.Operating_Cash_Flow_Margin_Rank AS FY2025_Operating_Cash_Flow_Margin_Rank,
    pb.Current_Ratio_Rank AS FY2025_Current_Ratio_Rank,
    pb.Debt_to_Equity_Rank AS FY2025_Debt_to_Equity_Rank,
    pb.ROA_Rank AS FY2025_ROA_Rank,
    pb.ROE_Rank AS FY2025_ROE_Rank,
    pb.Average_Rank AS FY2025_Average_Rank
FROM financial_analytics.analytics.performance p
INNER JOIN latest
    ON p.Company = latest.Company
    AND p.Fiscal_Year = latest.Latest_Fiscal_Year
LEFT JOIN financial_analytics.analytics.growth_trend gt
    ON p.Company = gt.Company
    AND p.Fiscal_Year = gt.Fiscal_Year
LEFT JOIN financial_analytics.analytics.financial_health fh
    ON p.Company = fh.Company
    AND p.Fiscal_Year = fh.Fiscal_Year
LEFT JOIN financial_analytics.analytics.peer_benchmarking pb
    ON p.Company = pb.Company


## Growth and momentum

### Business question

Is the company growing, and is that growth accelerating or slowing?

### Approach

View: growth_trend

- Growth_Rate = (current revenue − prior revenue) / prior revenue
- Prior_Growth_Rate = previous year's growth rate
- Growth_Differential = current growth rate − prior growth rate


In [0]:
%sql
SELECT Company, fiscal_year, Revenue, Growth_Rate, Prior_Growth_Rate, Growth_Differential
FROM financial_analytics.analytics.growth_trend 
ORDER BY Company, Fiscal_Year;


Company,fiscal_year,Revenue,Growth_Rate,Prior_Growth_Rate,Growth_Differential
AMD,2021,1.6434E10,null,null,null
AMD,2022,2.3601E10,0.436,null,null
AMD,2023,2.268E10,-0.039,0.436,-0.475
AMD,2024,2.5785E10,0.137,-0.039,0.176
AMD,2025,3.4639E10,0.343,0.137,0.206
Adobe,2021,1.5785E10,null,null,null
Adobe,2022,1.7606E10,0.115,null,null
Adobe,2023,1.9409E10,0.102,0.115,-0.013
Adobe,2024,2.1505E10,0.108,0.102,0.006
Adobe,2025,2.3769E10,0.105,0.108,-0.003


In [0]:
%sql
SELECT 
Company,
fiscal_year,
Growth_Differential,
Growth_Rate
FROM financial_analytics.analytics.growth_trend
WHERE fiscal_year >=2024
ORDER BY company


Company,fiscal_year,Growth_Differential,Growth_Rate
AMD,2024,0.176,0.137
AMD,2025,0.206,0.343
Adobe,2024,0.006,0.108
Adobe,2025,-0.003,0.105
Apple,2024,0.048,0.02
Apple,2025,0.044,0.064
CrowdStrike,2024,-0.181,0.363
CrowdStrike,2025,-0.069,0.294
CrowdStrike,2026,-0.077,0.217
Microsoft,2024,0.088,0.157


## Profitability

### Business Question

Is growth turning into profitable operations?

### Approach

View: `performance`

- Gross_Margin_pct = Gross Profit / Revenue
- Operating_Margin_pct = Operating Income / Revenue
- Net_Income_Margin_pct = Net Income / Revenue
- Net_Income_Growth and Operating_Income_Growth stay in the view as calculated percents.
- A percent is shown only when both years are profits. A wider loss is labeled "loss widened," a smaller loss is labeled "loss narrowed," and a move between a profit and a loss is labeled "not meaningful."
- CrowdStrike FY2026 net income is a wider loss, from 15.2 million to 162.5 million. The calculated ratio is 9.662 because the prior year was also a loss. That percent is not a growth rate to read.


In [0]:
%sql
SELECT Company, Fiscal_Year,Gross_Margin_pct ,Net_Income_Margin_pct ,Operating_Margin_pct,Operating_Income_Growth
FROM financial_analytics.analytics.performance
Where Fiscal_Year >= 2024
ORDER BY company, Fiscal_Year;


Company,Fiscal_Year,Gross_Margin_pct,Net_Income_Margin_pct,Operating_Margin_pct,Operating_Income_Growth
AMD,2024,0.494,0.064,0.074,3.738
AMD,2025,0.495,0.125,0.107,0.944
Adobe,2024,0.89,0.259,0.313,0.014
Adobe,2025,0.893,0.3,0.366,0.291
Apple,2024,0.462,0.24,0.315,0.078
Apple,2025,0.469,0.269,0.32,0.08
CrowdStrike,2024,0.752,0.024,-0.006,-0.899
CrowdStrike,2025,0.75,-0.004,-0.029,5.081
CrowdStrike,2026,0.747,-0.034,-0.061,1.52
Microsoft,2024,0.698,0.36,0.446,0.236


## Operating Efficiency

### Business Question

Is the business scaling efficiently, or are operating costs growing faster than revenue?

### Approach

View: `performance`

- Operating_Expense_Growth vs prior year
- Operating_Expense_to_Revenue
- Revenue_Growth_Minus_Opex_Growth = unrounded revenue growth − unrounded operating-expense growth, then rounded to 3 decimals

If revenue growth is higher than opex growth, efficiency is improving. If opex growth is higher, that is cost pressure to investigate.


In [0]:
%sql
SELECT p.Company, p.Fiscal_Year, p.Revenue,gt.Growth_Rate,p.Operating_Expense_Growth, p.Operating_Expense_to_Revenue, p.Revenue_Growth_Minus_Opex_Growth
FROM financial_analytics.analytics.performance p
join financial_analytics.analytics.growth_trend gt
on p.company=gt.company and p.fiscal_year=gt.fiscal_year
where p.Fiscal_Year >= 2024
ORDER BY Company, Fiscal_Year;


Company,Fiscal_Year,Revenue,Growth_Rate,Operating_Expense_Growth,Operating_Expense_to_Revenue,Revenue_Growth_Minus_Opex_Growth
AMD,2024,2.5785E10,0.137,0.076,0.42,0.061
AMD,2025,3.4639E10,0.343,0.243,0.389,0.1
Adobe,2024,2.1505E10,0.108,0.192,0.577,-0.084
Adobe,2025,2.3769E10,0.105,0.009,0.526,0.096
Apple,2024,3.91035E11,0.02,0.048,0.147,-0.028
Apple,2025,4.16161E11,0.064,0.082,0.149,-0.018
CrowdStrike,2024,3.055555E9,0.363,0.265,0.758,0.098
CrowdStrike,2025,3.953624E9,0.294,0.33,0.779,-0.036
CrowdStrike,2026,4.812005E9,0.217,0.262,0.808,-0.045
Microsoft,2024,2.45122E11,0.157,0.07,0.251,0.087


## Cash Flow & Cash Conversion

### Business Question

Is reported performance supported by cash from operations?

### Approach

View: `performance`

- Operating_Cash_Flow_Margin_pct = OCF / Revenue
- Operating_Cash_Flow_Growth vs prior year
- Cash_Conversion_Ratio = OCF / Net Income (null if net income is 0). The raw ratio is kept.
- Cash_Conversion_Meaningful is false when net income is zero or negative, or when abs(OCF / Net Income) > 5. A ratio of exactly 5 stays meaningful.
- Where the flag is false, read the ratio as n/m. CrowdStrike FY2025 is the example: a small net loss against large operating cash flow.
- Free cash flow, operating cash flow minus capital expenditures, is calculated in the view. It is null when capital expenditures are missing, including NVIDIA FY2021. The company reads below use operating cash flow.


In [0]:
%sql
SELECT Company, Fiscal_Year, Net_Income, Cash_Flow_from_Operating_Activities, Operating_Cash_Flow_Margin_pct, Operating_Cash_Flow_Growth, Cash_Conversion_Ratio, Capital_Expenditures, Free_Cash_Flow, Free_Cash_Flow_Margin_pct
FROM financial_analytics.analytics.performance
where Fiscal_Year >= 2024
ORDER BY Company, Fiscal_Year;


Company,Fiscal_Year,Net_Income,Cash_Flow_from_Operating_Activities,Operating_Cash_Flow_Margin_pct,Operating_Cash_Flow_Growth,Cash_Conversion_Ratio,Capital_Expenditures,Free_Cash_Flow,Free_Cash_Flow_Margin_pct
AMD,2024,1.641E9,3.041E9,0.118,0.824,1.853,6.36E8,2.405E9,0.093
AMD,2025,4.335E9,7.709E9,0.223,1.535,1.778,9.74E8,6.735E9,0.194
Adobe,2024,5.56E9,8.056E9,0.375,0.103,1.449,1.83E8,7.873E9,0.366
Adobe,2025,7.13E9,1.0031E10,0.422,0.245,1.407,1.79E8,9.852E9,0.414
Apple,2024,9.3736E10,1.18254E11,0.302,0.07,1.262,9.447E9,1.08807E11,0.278
Apple,2025,1.1201E11,1.11482E11,0.268,-0.057,0.995,1.2715E10,9.8767E10,0.237
CrowdStrike,2024,7.2181E7,1.166207E9,0.382,0.239,16.157,1.76529E8,9.89678E8,0.324
CrowdStrike,2025,-1.5241E7,1.381727E9,0.349,0.185,-90.659,2.54852E8,1.126875E9,0.285
CrowdStrike,2026,-1.62502E8,1.612349E9,0.335,0.167,-9.922,3.02108E8,1.310241E9,0.272
Microsoft,2024,8.8136E10,1.18548E11,0.484,0.354,1.345,4.4477E10,7.4071E10,0.302


## Financial Health & Risk

### Business Question

Does the company have the financial flexibility to support operations and growth if conditions tighten?

### Approach

View: `financial_health`

- Current_ratio
- debt_to_equity_ratio still treats a missing debt leg as 0. Short_Term_Debt_Reported and Long_Term_Debt_Reported record whether the source had a value.
- CrowdStrike short-term debt is unreported for FY2021 through FY2026 and is treated as 0.
- Liabilities_to_Assets. AMD null total liabilities are assets minus equity, because AMD companyfacts has no Total_Liabilities tag. Total_Liabilities_Imputed marks those rows. Other companies are not imputed.
- Primary ROA and ROE use average balances, (current + prior) / 2. Ending balances are used only when there is no prior year, and those rows are flagged.
- ROA_Ending and ROE_Ending keep the ending-balance ratios.
- Buybacks inflate Apple's ROE because they shrink stockholders' equity.


In [0]:
%sql
SELECT company,Fiscal_Year,Current_ratio,debt_to_equity_ratio,Liabilities_to_Assets,roa,roe
FROM financial_analytics.analytics.financial_health
where Fiscal_Year >=2024
ORDER BY company, Fiscal_Year;


company,Fiscal_Year,Current_ratio,debt_to_equity_ratio,Liabilities_to_Assets,roa,roe
AMD,2024,2.616,0.03,0.168,0.024,0.029
AMD,2025,2.85,0.051,0.181,0.056,0.069
Adobe,2024,1.068,0.399,0.533,0.184,0.394
Adobe,2025,0.996,0.534,0.606,0.242,0.613
Apple,2024,0.867,1.697,0.844,0.257,1.646
Apple,2025,0.893,1.23,0.795,0.312,1.519
CrowdStrike,2024,1.764,0.322,0.648,0.011,0.031
CrowdStrike,2025,1.766,0.227,0.619,-0.002,-0.005
CrowdStrike,2026,1.773,0.168,0.597,-0.015,-0.037
Microsoft,2024,1.275,0.167,0.476,0.172,0.328


## Peer Benchmarking

### Business question

How do the companies compare on the latest shared fiscal year?

### Approach

`peer_benchmarking` ranks nine FY2025 metrics with RANK(). FY2025 is the last year present for every company. The value is kept next to each rank.

Higher is better for revenue growth, operating margin, net income margin, OCF margin, current ratio, ROA, and ROE. Lower is better for operating expense / revenue and debt-to-equity. ROA and ROE are the primary average-balance returns.

Ties: RANK() shares the best rank and skips the next rank (pandas rank method='min').

Missing values: NULLS LAST, so a missing metric ranks after every reported value. Missing values tie with each other.

Average_Rank is the equal-weight average of those nine ranks. It is context for the section, not a single winner score. The six companies are in different technology subsectors and do not share one business model, so the average is a descriptive comparison, not a ranking of which company is best. Do not mix FY2025 and FY2026 in one ranking.

`executive_summary` attaches these FY2025 ranks on company alone, including when a company's latest year is FY2026. The columns are labeled FY2025_.


In [0]:
%sql
SELECT *
FROM financial_analytics.analytics.peer_benchmarking
ORDER BY Average_Rank


Company,Fiscal_Year,Growth_Rate,Growth_Rate_Rank,Operating_Margin_pct,Operating_Margin_Rank,Net_Income_Margin_pct,Net_Income_Margin_Rank,Operating_Expense_to_Revenue,Operating_Expense_to_Revenue_Rank,Operating_Cash_Flow_Margin_pct,Operating_Cash_Flow_Margin_Rank,Current_ratio,Current_Ratio_Rank,debt_to_equity_ratio,Debt_to_Equity_Rank,ROA,ROA_Rank,ROE,ROE_Rank,Average_Rank
NVIDIA,2025,1.142,1,0.624,1,0.558,1,0.126,1,0.491,1,4.44,1,0.107,2,0.822,1,1.192,2,1.222
Microsoft,2025,0.149,4,0.456,2,0.361,2,0.232,3,0.483,2,1.353,4,0.126,3,0.18,4,0.333,4,3.111
AMD,2025,0.343,2,0.107,5,0.125,5,0.389,4,0.223,6,2.85,2,0.051,1,0.059,5,0.072,5,3.889
Adobe,2025,0.105,5,0.366,3,0.3,3,0.526,5,0.422,3,0.996,5,0.534,5,0.239,3,0.554,3,3.889
Apple,2025,0.064,6,0.32,4,0.269,4,0.149,2,0.268,5,0.893,6,1.23,6,0.309,2,1.714,1,4.000
CrowdStrike,2025,0.294,3,-0.029,6,-0.004,6,0.779,6,0.349,4,1.766,3,0.227,4,-0.002,6,-0.005,6,4.889


## Executive summary

This comparison uses each company's own latest reported year. Microsoft, NVIDIA, and CrowdStrike are FY2026. Apple, Adobe, and AMD are FY2025. Those years do not cover the same months. NVIDIA and CrowdStrike end in January, Microsoft ends in June, Apple ends in September, and Adobe and AMD end near the calendar year. FY2025 is the last year available for all six, so the peer ranks stay on FY2025. A lower average rank is a stronger result on that nine-metric set. It is context for the comparison. It is not a ranking of which company is best.

NVIDIA and Microsoft are the strongest operating stories in their latest year, for different reasons. NVIDIA still turns 60.4% of revenue into operating profit, and operating cash flow grew 60.3%, even though growth and margins came down from FY2025. Microsoft grew revenue faster than in FY2025, raised its operating margin, and raised its operating-cash-flow margin from 48.3% to 55.1%. Adobe also converted modest revenue growth into a higher operating margin and higher operating cash flow, because operating expenses barely grew.

AMD is improving from a lower profit level. Revenue growth, operating margin, and operating cash flow all rose in FY2025. The operating margin is 10.7%, which is better than 7.4% a year earlier and still well below Microsoft, Apple, NVIDIA, and Adobe.

Three results deserve attention even though revenue did not fall. Apple's net income grew 19.5% while operating cash flow fell 5.7%. Receivables, inventory, payables, stock-based compensation, and depreciation are already in the file, and they do not close that gap. Adobe's profit and cash improved while the current ratio moved from 1.068 to 0.996 and debt-to-equity rose from 0.399 to 0.534. Cash fell from 7.6 billion to 5.4 billion, and long-term debt rose from 4.1 billion to 6.2 billion. CrowdStrike is still growing and still generating operating cash flow, and the operating loss widened from 116.4 million to 293.3 million. Stock-based compensation was 1.097 billion. That is a large non-cash expense, and it is not the whole reason operating cash flow stayed positive.

Revenue growth does not explain these differences. NVIDIA grew 65.5% and CrowdStrike grew 21.7%, and both grew more slowly than the year before. Microsoft grew faster, at 17.8%. Apple grew 6.4%. The useful comparison is whether growth is showing up in operating profit, whether operating expenses are growing slower than revenue, and whether earnings and operating cash are moving in the same direction.

## Individual company analysis

Each section below uses that company's latest fiscal year. The six parts separate what the statements show from what is worth a closer look. A change in a ratio is not treated as a proven cause.


### Microsoft — FY2026

Operating efficiency and cash generation.

**Financial performance.** Revenue growth was 17.8% in FY2026, up from 14.9% in FY2025. Gross margin eased from 68.8% to 67.9%. Operating margin rose from 45.6% to 46.8%, and net margin rose from 36.1% to 40.3%. Operating expenses grew 7.4%, against revenue growth of 17.8%. Operating cash flow grew 34.4%, and the operating-cash-flow margin rose from 48.3% to 55.1%.

**Key findings.** Gross margin and operating margin moved in different directions. Microsoft kept slightly less of each sales dollar after cost of revenue, then kept more after operating expenses. Operating expenses were 21.2% of revenue, down from 23.2% in FY2025 and 25.1% in FY2024. Revenue growth exceeded operating-expense growth by 10.3 percentage points. Cash generation improved by more than the operating margin. The operating-cash-flow margin gained 6.8 percentage points. The operating margin gained 1.2.

**Business implications.** The higher operating margin did not come from a higher gross margin. It came from operating expenses growing slower than revenue. That efficiency can fade if expense growth catches up, or if gross margin keeps easing. The cash result matters because operating cash flow grew faster than both revenue and net income. The statements show those outcomes. They do not show which products or cost lines produced them.

**Management recommendation.** Gross margin and the expense ratio moved in opposite directions, so they need a separate look. The company-year totals do not say whether the gross-margin dip is mix or delivery cost, or whether expenses can stay at 21.2% of revenue as sales keep growing.

**What to monitor.** Gross margin, operating expenses as a share of revenue, the gap between revenue growth and operating-expense growth, operating margin, and the operating-cash-flow margin.

**Further investigation.** Segment or product gross margin, and the operating-expense categories that grew more slowly than revenue. The company-year totals do not identify those lines.

### Apple — FY2025

Earnings and cash flow.

**Financial performance.** Revenue growth improved to 6.4% from 2.0% in FY2024. Gross margin rose from 46.2% to 46.9%, and operating margin rose from 31.5% to 32.0%. Net income grew 19.5%, from 93.7 billion to 112.0 billion, and the net margin rose from 24.0% to 26.9%. Operating cash flow moved the other way. It fell 5.7%, from 118.3 billion to 111.5 billion, and the operating-cash-flow margin fell from 30.2% to 26.8%. Cash conversion fell from 1.262 to 0.995. The current ratio was 0.893, still below 1, and debt-to-equity fell from 1.70 to 1.23.

**Key findings.** The file already has the accounts that a first pass at this gap would use. Accounts receivable rose from 33.4 billion to 39.8 billion. Inventory fell from 7.3 billion to 5.7 billion. Accounts payable rose from 69.0 billion to 69.9 billion. Those year-end changes do not point the same way. Higher receivables use cash if the increase is operating, lower inventory releases cash, and higher payables are a source of cash. Taken together, the three changes are a net use of about 3.9 billion. In FY2024 the same three changes were a net source of about 1.5 billion.

Stock-based compensation rose from 11.7 billion to 12.9 billion, and depreciation and amortization rose from 11.4 billion to 11.7 billion. Both are usually added back in the reconciliation from net income to operating cash flow. The increases work against an explanation of the cash decline.

A simple sum of FY2025 net income, those two add-backs, and the 3.9 billion net use comes to about 132.7 billion. Reported operating cash flow was 111.5 billion, about 21.2 billion lower. The same sum in FY2024 was within 107 million of operating cash flow. That comparison is useful, and it is not the cash-flow statement. Year-end balances are not the change lines in the operating section. Other current liabilities also moved by large amounts, up about 19.5 billion in FY2024 and down about 11.9 billion in FY2025. Putting that liability change into the same sum does not close FY2025, and it opens a large gap in FY2024, the year the smaller set of items had nearly matched. None of these accounts can be named as the cause of the cash decline.

**Business implications.** Apple was more profitable in FY2025 and generated less operating cash. Cash conversion near 1.0 can be a reasonable result. The issue is the direction: conversion fell in the year earnings grew. Operating expenses grew faster than revenue by 1.7 percentage points, which is small next to the cash gap. The current ratio below 1 is a liquidity signal that was already present in FY2024. Average-balance ROE of 171.4% should be read with average-balance ROA of 30.9%, because buybacks shrink equity.

**Management recommendation.** The earnings increase is not a cash increase. Higher stock-based compensation, higher depreciation, and the year-end move in receivables, inventory, and payables do not explain the cash decline. Those amounts are already in the file. The next place to look is the cash-flow statement's own operating section.

**What to monitor.** Net income, operating cash flow, cash conversion, the operating-cash-flow margin, and the year-end receivables, inventory, and payables balances. The current ratio is the liquidity check.

**Further investigation.** The remaining operating cash-flow lines, including changes in operating assets and liabilities that this extract does not separate. Until those lines are in hand, the 21.2 billion gap between the simple sum and reported operating cash flow stays unexplained.


### NVIDIA — FY2026

Growth and margin performance.

**Financial performance.** Revenue grew 65.5% in FY2026, down from 114.2% in FY2025. Gross margin was 71.1%, down from 75.0%. Operating margin was 60.4%, down from 62.4% and still above 54.1% in FY2024. Net margin was 55.6%, compared with 55.8%. Operating expenses grew 40.7%, slower than revenue, and fell from 12.6% of revenue to 10.7%. Operating cash flow grew 60.3%. The operating-cash-flow margin was 47.6%, down from 49.1%.

**Key findings.** Growth is still high, and it is slower than an unusually strong FY2025. Revenue did not decline. Margins eased from FY2025 levels and remain high. Operating expenses grew more slowly than revenue, so the expense ratio is still falling. Average-balance ROA eased from 82.2% to 75.4% and is still very high. The balance sheet is not the constraint in this comparison. The current ratio is 3.905, and debt-to-equity fell from 0.107 to 0.054.

**Business implications.** NVIDIA is still converting most of its revenue into operating profit and a large share into operating cash. The FY2026 question is whether those margins depended on the FY2025 growth rate. A lower growth rate can come with lower margins without meaning demand has turned down. This dataset shows the moderation. It does not show orders, volume, or price.

**Management recommendation.** Read FY2026 against a more ordinary growth rate, not against 114.2%. The open question is whether gross margin, operating margin, and the operating-cash-flow margin stay near these levels if growth cools further. Slower growth is not weaker demand unless volume or price says so.

**What to monitor.** Revenue growth, gross margin, operating margin, operating expenses as a share of revenue, and the operating-cash-flow margin.

**Further investigation.** How much of the gross-margin change is price, cost, or mix. That split is not in the company-year totals.

### Adobe — FY2025

Operating efficiency and financial health.

**Financial performance.** Revenue grew 10.5%, compared with 10.8% in FY2024. Gross margin was 89.3%, compared with 89.0%. Operating margin rose from 31.3% to 36.6%, and net margin rose from 25.9% to 30.0%. Operating expenses grew 0.9%, against revenue growth of 10.5%, and fell from 57.7% of revenue to 52.6%. Operating cash flow grew 24.5%. The operating-cash-flow margin rose from 37.5% to 42.2%. The current ratio fell from 1.068 to 0.996. Debt-to-equity rose from 0.399 to 0.534, and liabilities-to-assets rose from 53.3% to 60.6%.

**Key findings.** Profitability and cash generation are the strong part of the year. Gross margin barely moved, so the operating-margin increase came from the lower expense ratio. The balance sheet moved the other way. A current ratio of 0.996 means current liabilities slightly exceed current assets. That is a liquidity signal, not proof of financial distress. Current assets fell from 11.2 billion to 10.2 billion, and current liabilities fell from 10.5 billion to 10.2 billion, so the ratio slipped below 1 because assets fell by more. Cash fell from 7.6 billion to 5.4 billion, while short-term investments rose from 0.3 billion to 1.2 billion. Short-term debt went from 1.5 billion to zero, and long-term debt rose from 4.1 billion to 6.2 billion. Share repurchases were 11.3 billion, compared with 9.5 billion. Leverage is higher than a year earlier, and the debt increase is visible in the long-term balance. Returns are higher too: average-balance ROA rose from 18.5% to 23.9%.

**Business implications.** Modest revenue growth still lifted operating profit because expenses grew much more slowly than sales. The efficiency gain and the tighter liquidity are in the same year. One does not cancel the other. Management can keep the operating improvement and still need a clear view of cash, near-term obligations, and the larger long-term debt balance.

**Management recommendation.** The expense ratio fell to 52.6% of revenue because spending barely grew. That is the part to test: can it stay there. Cash and debt are a separate issue. Cash is lower, short-term debt is gone, and long-term debt is higher. A current ratio of 0.996 is a liquidity flag, not a crisis.

**What to monitor.** Operating margin, operating expenses as a share of revenue, the operating-cash-flow margin, cash, the current ratio, and debt-to-equity.

**Further investigation.** The maturity schedule for the debt, and how much of the cash decline is the 11.3 billion of share repurchases versus other uses. The direction of cash, short-term investments, and debt is already in the file.


### AMD — FY2025

Revenue growth and profitability.

**Financial performance.** Revenue growth accelerated from 13.7% in FY2024 to 34.3% in FY2025. Gross margin was almost unchanged, 49.4% then 49.5%. Operating margin rose from 7.4% to 10.7%. Net margin rose from 6.4% to 12.5%. Operating income grew 94.4%. Operating expenses grew 24.3%, behind revenue, and fell from 42.0% of revenue to 38.9%. Operating cash flow grew 153.5%, and the operating-cash-flow margin rose from 11.8% to 22.3%.

**Key findings.** Growth, operating profit, and cash generation improved together. Gross margin did not. The operating-margin gain came from expenses growing slower than revenue. The level still matters. An operating margin of 10.7% and average-balance ROA of 5.9% are better than FY2024 and still well below Microsoft, Apple, NVIDIA, and Adobe. Liabilities-to-assets rose from 16.8% to 18.1%. AMD does not report total liabilities, so that ratio is assets minus equity. The current ratio improved from 2.62 to 2.85. Debt-to-equity rose from 0.030 to 0.051 and is still low.

**Business implications.** AMD is converting faster revenue growth into higher operating profit and much higher operating cash flow. The open question is how far that conversion can go. Faster growth does not, by itself, close the operating-margin gap with the more profitable companies in this set. The liability ratio moved up from a low base. Because liabilities are imputed, that 1.3 percentage-point increase should not be treated as a financing decision.

**Management recommendation.** Watch whether revenue growth keeps lifting an operating margin that is still only 10.7%. The consolidated totals do not show expense mix or product-level profit, so they do not support a specific cost cut.

**What to monitor.** Revenue growth, operating margin, operating expenses as a share of revenue, the operating-cash-flow margin, and average-balance ROA.

**Further investigation.** Operating-expense composition and profitability by product. A leverage review also needs AMD's reported total liabilities. That row is missing from the clean facts table.

### CrowdStrike — FY2026

Operating profitability and cash generation.

**Financial performance.** Revenue grew 21.7%, down from 29.4% in FY2025 and 36.3% in FY2024. Revenue is still growing. Gross margin was 74.7%, compared with 75.0% and 75.2% in the two prior years. The operating margin was -6.1%, after -2.9% and -0.6%. The operating loss widened from 116.4 million to 293.3 million. Net income went from a loss of 15.2 million to a loss of 162.5 million, and the net margin went from -0.4% to -3.4%. Operating expenses grew 26.2%, ahead of revenue by 4.5 percentage points, and reached 80.8% of revenue. Operating cash flow still grew 16.7%, to 1.612 billion. The operating-cash-flow margin was 33.5%, down from 34.9%.

**Key findings.** The wider loss is not a gross-margin story. Gross margin stayed near 75%. Operating expenses are the pressure, and the file already splits them. Research and development rose from 1.076 billion to 1.385 billion, and general and administrative rose from 481 million to 670 million. Both grew faster than revenue. Sales and marketing rose from 1.523 billion to 1.831 billion, or 20.2%, a little slower than revenue growth of 21.7%. Those three categories add up to operating expenses.

Operating cash flow was 1.612 billion in a year when net income was a 162.5 million loss. The calculated net-income growth rate is 9.662, or about 966%. That percent is not meaningful. Both years are losses, and the loss got larger, so the display for this year is "loss widened." The same issue applies to operating income. The calculated growth rate is positive because the prior year was also a loss.

Stock-based compensation was 1.097 billion, up from 861 million. That is 22.8% of revenue and about 68% of operating cash flow. It is a non-cash expense, so it reduces GAAP earnings and is added back in operating cash flow. It is an important part of the gap and not the whole gap. Operating cash flow exceeded net income by 1.775 billion. Stock-based compensation accounts for 1.097 billion of that difference, and about 678 million remains. Depreciation and amortization is missing for CrowdStrike in this extract. It is not part of the explanation, and it is not treated as zero.

**Business implications.** Positive operating cash flow does not mean CrowdStrike is GAAP profitable. GAAP operating income is a loss of 293.3 million, and GAAP net income is a loss of 162.5 million. Cash from operations can stay positive when non-cash expenses and balance-sheet changes are larger than the accounting loss. The spending pattern is separate from that cash result. Operating expenses grew faster than revenue, so the operating loss widened even though sales grew 21.7%. The current ratio was 1.773 and debt-to-equity fell to 0.168. Liquidity and leverage are not the issue in this comparison.

**Management recommendation.** Research and development and general and administrative grew faster than the 21.7% sales growth. Sales and marketing did not. Read the 1.612 billion of operating cash flow next to the GAAP loss. Stock-based compensation is a large part of why cash and earnings differ, and about 678 million of that gap is still unexplained.

**What to monitor.** Revenue growth, operating margin, operating expenses as a share of revenue, the growth of research and development and general and administrative, stock-based compensation relative to operating cash flow, and the operating-cash-flow margin.

**Further investigation.** The rest of the operating-section reconciliation. Current liabilities rose from 3.46 billion to 4.18 billion while accounts payable fell, so the increase sits in a liability line this file does not name. That line is not established here as the reason operating cash flow is positive. Depreciation and amortization is also unreported. The expense categories above are enough to see where spending grew faster than sales. They do not show whether that spending is still buying growth.


## Cross-company comparison

These companies do not share one business model, and their fiscal years do not cover the same months. The point of the comparison is how the latest results differ, and why that leads to different management work. It is not a ranking from best to worst.

**Revenue growth into operating profit.** Microsoft, Adobe, and AMD all raised operating margin while revenue grew, and in each case operating expenses grew more slowly than revenue. Microsoft's operating margin rose from 45.6% to 46.8% with revenue growth of 17.8% and expense growth of 7.4%. Adobe's rose from 31.3% to 36.6% with expense growth of 0.9%. AMD's rose from 7.4% to 10.7%. That is a real improvement, and 10.7% is still the lowest operating margin among the companies here that are profitable. NVIDIA's operating margin eased from 62.4% to 60.4% while revenue grew 65.5% and expenses grew 40.7%. Profitability stayed very high. It did not improve. CrowdStrike grew revenue 21.7% and operating expenses 26.2%, and the operating margin fell from -2.9% to -6.1%.

**Operating-expense efficiency.** Adobe and Microsoft show the clearest gap between sales growth and expense growth. AMD's expense ratio also fell, from 42.0% of revenue to 38.9%, from a higher starting point. NVIDIA's expense ratio is already low, at 10.7% of revenue. Apple moved the other way by a small amount: expense growth exceeded revenue growth by 1.7 percentage points. CrowdStrike's expense ratio rose for a second year, to 80.8%.

**Earnings and cash.** Apple is the clear split between reported earnings and operating cash. Net income grew 19.5% and operating cash flow fell 5.7%, from 118.3 billion to 111.5 billion, so cash conversion fell from 1.262 to 0.995. The year-end change in receivables, inventory, and payables was a net use of about 3.9 billion, and stock-based compensation and depreciation both increased. Those items do not close the gap. Microsoft's operating cash flow grew 34.4% against revenue growth of 17.8%, so cash generation improved faster than sales. Earnings and operating cash moved together there, and at NVIDIA and AMD as well. CrowdStrike is a different split. Operating cash flow grew 16.7% while the GAAP operating loss got larger. Stock-based compensation of 1.097 billion is a large non-cash expense in that result, about 68% of operating cash flow, and about 678 million of the gap between operating cash flow and net income is still unexplained. Positive operating cash flow there is not evidence of GAAP profit.

**Balance-sheet items that need another look.** Adobe's current ratio moved from 1.068 to 0.996 and debt-to-equity from 0.399 to 0.534 in a year when profit and cash improved. Cash fell from 7.6 billion to 5.4 billion, short-term debt went from 1.5 billion to zero, and long-term debt rose from 4.1 billion to 6.2 billion. Apple's current ratio stayed below 1, at 0.893, while debt-to-equity improved from 1.70 to 1.23. AMD's liabilities-to-assets rose from 16.8% to 18.1% on imputed liabilities, from a low base, and the current ratio is 2.85. NVIDIA and Microsoft do not have a balance-sheet result that competes with the operating story. CrowdStrike's current ratio and lower debt-to-equity are not the issue. The operating loss is.

**Why the priorities differ.** For Microsoft, gross margin and the expense ratio moved in opposite directions, and the totals do not say whether expenses can stay at 21.2% of revenue. For NVIDIA, FY2026 should be read against a more ordinary growth rate, not against 114.2%. For AMD, the question is whether faster growth keeps lifting an operating margin that is still only 10.7%. For Apple, the earnings increase is not a cash increase, and the accounts already in the file do not explain the drop. For Adobe, the expense ratio and the lower cash balance are two different questions. For CrowdStrike, positive operating cash flow is not a GAAP profit, and stock-based compensation is not the whole gap.


## Management attention summary

One priority for each company. The evidence is the latest reported year. The next look is what to examine, not a cause the statements already proved.

| Company | Main financial signal | Supporting evidence | Business implication | Next look | KPIs to monitor | Additional information needed |
|---|---|---|---|---|---|---|
| Microsoft | Operating margin and cash generation improved while gross margin eased | Revenue growth 17.8% vs 14.9%. Gross margin 67.9% vs 68.8%. Operating margin 46.8% vs 45.6%. Operating expenses grew 7.4%. Operating-cash-flow margin 55.1% vs 48.3%. | The operating-margin gain came from slower expense growth, not from a higher gross margin. | Gross margin and the expense ratio moved in opposite directions. The totals do not say which products or cost lines did it, or whether expenses can stay at 21.2% of revenue. | Gross margin, operating expense / revenue, operating margin, operating-cash-flow margin | Which products or cost lines drove the gross-margin change and the lower expense ratio |
| Apple | Net income rose and operating cash flow fell | Net income +19.5%, from 93.7 billion to 112.0 billion. Operating cash flow -5.7%, from 118.3 billion to 111.5 billion. Cash conversion 0.995 vs 1.262. Receivables +6.4 billion, inventory -1.6 billion, payables +0.9 billion. Stock-based compensation 11.7 billion to 12.9 billion. Depreciation 11.4 billion to 11.7 billion. | The add-backs increased, and the three working-capital accounts were a net use of about 3.9 billion. Net income, the two add-backs, and that use are still about 21.2 billion above operating cash flow. Year-end balances do not name the cause. | The earnings increase is not a cash increase. The accounts already in the file do not explain the drop. The next place to look is the cash-flow statement's own operating section. | Net income, operating cash flow, cash conversion, receivables, inventory, payables | The remaining operating-section lines. The accounts above are already in the file, and they do not close the gap. |
| NVIDIA | Growth and margins cooled from a very strong prior year and remain high | Revenue growth 65.5% vs 114.2%. Gross margin 71.1% vs 75.0%. Operating margin 60.4%. Operating expenses grew 40.7%. Operating cash flow +60.3%. | Slower growth is not a drop in revenue. The question is whether current profitability holds as growth normalizes. | Read FY2026 against a more ordinary growth rate, not only against 114.2%. Slower growth is not weaker demand unless volume or price says so. | Revenue growth, gross margin, operating margin, operating-cash-flow margin | How much of the gross-margin change is price, cost, or mix |
| AMD | Faster revenue growth is lifting operating profit from a lower base | Revenue growth 34.3% vs 13.7%. Operating margin 10.7% vs 7.4%. Operating income +94.4%. Operating cash flow +153.5%. Operating-cash-flow margin 22.3% vs 11.8%. | Growth is converting into higher operating profit. The operating margin is still much lower than several peers. | Watch whether revenue growth keeps lifting an operating margin that is still only 10.7%. The totals do not support a specific cost cut. | Revenue growth, operating margin, operating expense / revenue, operating-cash-flow margin | Expense composition, product profitability, and reported total liabilities. The ratio here is imputed. |
| Adobe | Operating efficiency improved while liquidity tightened and leverage rose | Operating margin 36.6% vs 31.3%. Operating expenses grew 0.9% vs revenue growth of 10.5%. Expense ratio 52.6% vs 57.7%. Operating-cash-flow margin 42.2%. Current ratio 0.996. Debt-to-equity 0.534 vs 0.399. Cash 7.6 billion to 5.4 billion. Long-term debt 4.1 billion to 6.2 billion. Short-term debt 1.5 billion to zero. | Profit and cash improved because expenses grew much more slowly than revenue. Current assets fell by more than current liabilities. A current ratio just under 1 is a liquidity signal, not proof of distress. | Test whether spending can stay near 52.6% of revenue. Cash and debt are a separate look: cash is lower and long-term debt is higher. | Operating margin, operating expense / revenue, operating-cash-flow margin, cash, current ratio, debt-to-equity | The debt maturity schedule, and how much of the cash decline is share repurchases versus other uses. |
| CrowdStrike | Revenue and operating cash flow are still positive, and the GAAP loss widened | Revenue growth 21.7% vs 29.4%. Operating margin -6.1% vs -2.9%. Operating loss 116.4 million to 293.3 million. Net loss 15.2 million to 162.5 million. Research and development and general and administrative grew faster than revenue. Sales and marketing grew 20.2%. Operating cash flow 1.612 billion. Stock-based compensation 1.097 billion, about 68% of operating cash flow. | Positive operating cash flow is not GAAP profit. Stock-based compensation is a large non-cash expense and not the whole gap. About 678 million of the difference between operating cash flow and net income remains unexplained. The 966% net-income figure is a wider loss, not growth. | R&D and general and administrative grew faster than sales. Read operating cash flow next to the GAAP loss. Stock-based compensation is not the whole gap. | Revenue growth, operating margin, operating expense / revenue, stock-based compensation, operating-cash-flow margin | The rest of the operating cash-flow reconciliation, including the current liability that rose while accounts payable fell. Depreciation and amortization is unreported and is not treated as zero. |

These recommendations use the historical financial statements already in the project. They are not forecasts. Where the statements show a movement and not a cause, the next step is to get the missing detail rather than to invent an explanation.
